In [2]:
!pip install scikit_posthocs

In [5]:
"""
Prueba de Friedman para comparación de modelos de series temporales
Arquitecturas: Conv1D_Original | MLP_Light_Dropout | LSTM_Original
Métricas de CV: R², MAE, MAPE por fold (10 folds)

La prueba de Friedman es el equivalente no paramétrico del ANOVA de medidas repetidas.
Es ideal para comparar k tratamientos (modelos) en n bloques (folds).
H0: Todos los modelos tienen el mismo rendimiento esperado.
"""

# /// script
# requires-python = ">=3.10"
# dependencies = [
#   "scipy>=1.11",
#   "numpy>=1.24",
#   "pandas>=2.0",
#   "matplotlib>=3.7",
#   "seaborn>=0.12",
#   "scikit-posthocs>=0.7",
# ]
# ///

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. DATOS DE CV — extraídos de los logs
# ─────────────────────────────────────────────

# R² por fold (mean_R2 de cada fold)
r2_conv1d = np.array([
    0.946924, 0.948780, 0.940760, 0.936671, 0.959726,
    0.907311, 0.939827, 0.965339, 0.972436, 0.963541
])

r2_mlp = np.array([
    0.606714,  0.299086,  0.515690,  0.610961,  0.579360,
   -0.540573,  0.270211,  0.445612, -0.326927, -0.296392
])

r2_lstm = np.array([
    0.618736,  0.500013,  0.511257,  0.564208,  0.631297,
   -0.327178,  0.549634,  0.326734, -0.202341, -0.093617
])

# MAE por fold
mae_conv1d = np.array([
    0.786294, 0.872292, 0.908926, 0.902798, 0.839439,
    0.934090, 0.885430, 0.704127, 0.568145, 0.595615
])

mae_mlp = np.array([
    2.305112, 3.882513, 3.203850, 2.680821, 3.330805,
    3.626541, 3.032172, 3.141564, 5.741864, 4.121177
])

mae_lstm = np.array([
    2.077956, 3.163578, 3.042879, 2.796658, 2.890074,
    3.736644, 2.510860, 3.514077, 4.668801, 3.641706
])

# MAPE por fold (en %)
mape_conv1d = np.array([
    1.7617, 2.1360, 2.1745, 2.4769, 2.0841,
    2.0750, 2.0260, 1.6983, 1.3624, 1.3543
])

mape_mlp = np.array([
    4.9231,  8.2138,  7.1713,  6.5158,  7.7969,
    7.4688,  6.4518,  7.5206, 12.9377,  7.9219
])

mape_lstm = np.array([
    # calculados como promedio de las 4 variables por fold
    (2.24+4.85+4.73+6.21)/4,
    (3.26+9.08+4.77+10.43)/4,
    (2.87+8.77+4.79+10.11)/4,
    (2.52+6.53+8.50+10.10)/4,
    (3.29+6.45+8.37+10.50)/4,
    (2.74+8.52+7.12+11.79)/4,
    (3.68+4.44+7.48+8.52)/4,
    (4.56+8.71+8.23+11.68)/4,
    (7.06+7.12+9.74+21.55)/4,
    (4.46+7.02+8.94+10.41)/4,
])

# Gap (val_mae - train_mae) / train_mae por fold
gap_conv1d = np.array([
    0.053771, 0.270561, 0.328395, 0.349201, 0.381583,
    0.324124, 0.447071, 0.181649, -0.065208, -0.039862
])

gap_mlp = np.array([
    0.026675, 0.650726, 0.616925, 0.113923, 0.376001,
    0.444085, 0.228882, 0.454451, 1.523438, 0.785836
])

gap_lstm = np.array([
    0.538129, 2.087076, 2.110961, 1.565918, 1.530028,
    2.061255, 1.874306, 3.009461, 3.939087, 3.269680
])

MODELS = ["Conv1D", "MLP", "LSTM"]
N_FOLDS = 10

# ─────────────────────────────────────────────
# 2. FUNCIÓN PRUEBA DE FRIEDMAN + POST-HOC
# ─────────────────────────────────────────────

def friedman_analysis(data_matrix, metric_name, higher_is_better=True):
    """
    data_matrix: shape (n_folds, n_models)
    Retorna dict con resultados completos.
    """
    stat, p_value = stats.friedmanchisquare(*[data_matrix[:, i] for i in range(data_matrix.shape[1])])

    # Rangos de Friedman (promedio de rangos por modelo)
    n, k = data_matrix.shape
    # Rank cada fila (fold); si higher_is_better -> rank 1=peor, k=mejor
    # scipy usa ascendente por defecto
    ranked = np.apply_along_axis(
        lambda row: stats.rankdata(row if higher_is_better else -row),
        axis=1,
        arr=data_matrix
    )
    mean_ranks = ranked.mean(axis=0)

    # Post-hoc: Nemenyi (Wilcoxon signed-rank con corrección Bonferroni-Holm)
    df_for_posthoc = pd.DataFrame(data_matrix, columns=MODELS)
    nemenyi = sp.posthoc_nemenyi_friedman(df_for_posthoc)

    # Wilcoxon pairwise
    pairs = [("Conv1D","MLP"), ("Conv1D","LSTM"), ("MLP","LSTM")]
    wilcoxon_results = {}
    col_idx = {m: i for i, m in enumerate(MODELS)}
    for a, b in pairs:
        x = data_matrix[:, col_idx[a]]
        y = data_matrix[:, col_idx[b]]
        try:
            w_stat, w_p = stats.wilcoxon(x, y, alternative="two-sided")
        except Exception:
            w_stat, w_p = np.nan, np.nan
        wilcoxon_results[f"{a} vs {b}"] = {"statistic": w_stat, "p_value": w_p}

    return {
        "metric": metric_name,
        "friedman_stat": stat,
        "p_value": p_value,
        "significant": p_value < 0.05,
        "mean_ranks": dict(zip(MODELS, mean_ranks)),
        "means": dict(zip(MODELS, data_matrix.mean(axis=0))),
        "stds": dict(zip(MODELS, data_matrix.std(axis=0))),
        "nemenyi": nemenyi,
        "wilcoxon": wilcoxon_results,
        "data_matrix": data_matrix,
        "mean_ranks_array": mean_ranks,
    }


# ─────────────────────────────────────────────
# 3. EJECUTAR ANÁLISIS PARA CADA MÉTRICA
# ─────────────────────────────────────────────

metrics_data = {
    "R²":   (np.column_stack([r2_conv1d, r2_mlp, r2_lstm]),   True),
    "MAE":  (np.column_stack([mae_conv1d, mae_mlp, mae_lstm]), False),
    "MAPE": (np.column_stack([mape_conv1d, mape_mlp, mape_lstm]), False),
    "Gap":  (np.column_stack([gap_conv1d, gap_mlp, gap_lstm]), False),
}

results = {}
for metric_name, (matrix, hib) in metrics_data.items():
    results[metric_name] = friedman_analysis(matrix, metric_name, higher_is_better=hib)


# ─────────────────────────────────────────────
# 4. IMPRIMIR REPORTE COMPLETO EN CONSOLA
# ─────────────────────────────────────────────

def stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

print("=" * 72)
print("  PRUEBA DE FRIEDMAN — COMPARACIÓN DE MODELOS DE SERIES TEMPORALES")
print("  Conv1D_Original | MLP_Light_Dropout | LSTM_Original")
print("  10-fold Cross-Validation | 4 variables objetivo")
print("=" * 72)

for metric_name, res in results.items():
    print(f"\n{'─'*72}")
    print(f"  MÉTRICA: {metric_name}")
    print(f"{'─'*72}")
    print(f"  Estadístico de Friedman χ²  : {res['friedman_stat']:.4f}")
    print(f"  p-valor                     : {res['p_value']:.6f}  {stars(res['p_value'])}")
    print(f"  Significativo (α=0.05)      : {'SÍ ✔' if res['significant'] else 'NO ✗'}")
    print()
    print(f"  {'Modelo':<12}  {'Media':>10}  {'Std':>8}  {'Rango medio':>12}")
    print(f"  {'─'*50}")
    for m in MODELS:
        print(f"  {m:<12}  {res['means'][m]:>10.4f}  {res['stds'][m]:>8.4f}  {res['mean_ranks'][m]:>12.3f}")

    if res['significant']:
        print(f"\n  Post-hoc Nemenyi (p-valores):")
        nem = res['nemenyi']
        for i, a in enumerate(MODELS):
            for j, b in enumerate(MODELS):
                if j > i:
                    p = nem.loc[a, b]
                    print(f"    {a} vs {b}: p={p:.4f}  {stars(p)}")

        print(f"\n  Wilcoxon signed-rank (pairwise):")
        for pair, wres in res['wilcoxon'].items():
            p = wres['p_value']
            w = wres['statistic']
            print(f"    {pair}: W={w:.1f}, p={p:.4f}  {stars(p)}")
    else:
        print(f"\n  → No hay diferencias significativas entre modelos para {metric_name}.")

print(f"\n{'=' * 72}")
print("  INTERPRETACIÓN GLOBAL")
print(f"{'=' * 72}")

sig_metrics = [m for m, r in results.items() if r['significant']]
nonsig_metrics = [m for m, r in results.items() if not r['significant']]

if sig_metrics:
    print(f"\n  Métricas con diferencias significativas: {', '.join(sig_metrics)}")
    print()
    print("  Conclusión:")
    print("  Conv1D obtiene el rango de Friedman máximo (3.0/3.0) en R², MAE y MAPE,")
    print("  superando significativamente a MLP y LSTM en precisión (p<0.05 post-hoc).")
    print("  Además presenta el menor gap de overfitting entre modelos.")
    print()
    print("  🏆 Mejor modelo: Conv1D")

if nonsig_metrics:
    print(f"\n  Métricas sin diferencias significativas: {', '.join(nonsig_metrics)}")


# ─────────────────────────────────────────────
# 5. VISUALIZACIÓN
# ─────────────────────────────────────────────

COLORS = {
    "Conv1D": "#2E86AB",
    "MLP":    "#E84855",
    "LSTM":   "#F4A261",
}

plt.rcParams.update({
    "text.color": "white",
    "axes.labelcolor": "white",
    "axes.edgecolor": "#444",
    "xtick.color": "white",
    "ytick.color": "white",
    "axes.facecolor": "#1A1D2E",
    "figure.facecolor": "#0F1117",
    "axes.titlepad": 12,
})

metric_order = list(results.keys())

# ── Layout con GridSpec para control total de espaciado ──────────────
fig = plt.figure(figsize=(22, 30))
fig.patch.set_facecolor("#0F1117")

from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

# Fila 0: título global
# Fila 1: 4 boxplots (uno por métrica) en fila horizontal
# Fila 2: barras de rangos de Friedman (ancho completo)
# Fila 3: 4 heatmaps Nemenyi (uno por métrica)
# Fila 4: línea de evolución R²  +  tabla resumen

gs_main = GridSpec(
    5, 1,
    figure=fig,
    hspace=0.55,
    top=0.95, bottom=0.03,
    left=0.05, right=0.97,
    height_ratios=[0.04, 0.22, 0.20, 0.28, 0.22],
)

# ── Título global ────────────────────────────────────────────────────
ax_title = fig.add_subplot(gs_main[0])
ax_title.axis("off")
ax_title.text(
    0.5, 0.5,
    "Prueba de Friedman — Comparación de modelos de series temporales\n"
    "Conv1D vs MLP vs LSTM  ·  10-fold CV  ·  4 variables objetivo",
    ha="center", va="center", fontsize=16, fontweight="bold",
    color="white", transform=ax_title.transAxes,
)

# ── Fila 1: Boxplots ─────────────────────────────────────────────────
gs_box = GridSpecFromSubplotSpec(1, 4, subplot_spec=gs_main[1], wspace=0.38)

metric_ylabel = {
    "R²":   "R²",
    "MAE":  "MAE",
    "MAPE": "MAPE (%)",
    "Gap":  "Gap (val−train)/train",
}

for i, metric_name in enumerate(metric_order):
    ax = fig.add_subplot(gs_box[i])
    ax.set_facecolor("#1A1D2E")

    data_to_plot = []
    labels_bp   = []
    colors_bp   = []
    for m in MODELS:
        col_idx = MODELS.index(m)
        data_to_plot.append(metrics_data[metric_name][0][:, col_idx])
        labels_bp.append(m)
        colors_bp.append(COLORS[m])

    bp = ax.boxplot(
        data_to_plot, patch_artist=True, widths=0.5,
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(color="#888"),
        capprops=dict(color="#888"),
        flierprops=dict(markerfacecolor="#888", marker="o", markersize=4),
    )
    for patch, color in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)

    res = results[metric_name]
    sig_text = f"χ²={res['friedman_stat']:.2f}  p={res['p_value']:.4f} {stars(res['p_value'])}"

    ax.set_title(
        f"{metric_name} por modelo\n{sig_text}",
        color="white", fontsize=10, pad=6, linespacing=1.5,
    )
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(labels_bp, fontsize=9)
    ax.set_ylabel(metric_ylabel[metric_name], fontsize=9, color="#ccc")
    ax.grid(axis="y", alpha=0.3, color="#555")


# ── Fila 2: Rangos medios de Friedman ────────────────────────────────
ax_cd = fig.add_subplot(gs_main[2])
ax_cd.set_facecolor("#1A1D2E")

x_pos = np.arange(len(metric_order))
width = 0.22

for i, model in enumerate(MODELS):
    ranks_per_metric = [results[m]["mean_ranks"][model] for m in metric_order]
    bars = ax_cd.bar(
        x_pos + i * width, ranks_per_metric, width,
        label=model, color=COLORS[model], alpha=0.85,
        edgecolor="white", linewidth=0.5,
    )
    for bar, val in zip(bars, ranks_per_metric):
        ax_cd.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.04,
            f"{val:.2f}",
            ha="center", va="bottom", fontsize=9, color="white",
        )

ax_cd.set_xticks(x_pos + width)
ax_cd.set_xticklabels(metric_order, fontsize=11)
ax_cd.set_ylabel("Rango medio de Friedman", fontsize=10, color="#ccc")
ax_cd.set_title(
    "Rangos medios de Friedman por métrica\n"
    "R²: mayor rango = mejor  ·  MAE / MAPE / Gap: menor rango = mejor",
    color="white", fontsize=11,
)
ax_cd.legend(fontsize=10, facecolor="#1A1D2E", labelcolor="white", edgecolor="#555")
ax_cd.set_ylim(0, 3.8)
ax_cd.axhline(y=2.0, color="#555", linestyle="--", alpha=0.5)
ax_cd.grid(axis="y", alpha=0.25, color="#555")


# ── Fila 3: Heatmaps Nemenyi (uno por cada métrica significativa) ─────
sig_results = [(m, results[m]) for m in metric_order if results[m]["significant"]]
n_sig = len(sig_results)

gs_heat = GridSpecFromSubplotSpec(1, n_sig, subplot_spec=gs_main[3], wspace=0.5)
cmap_nemenyi = sns.diverging_palette(10, 130, as_cmap=True)

for idx_s, (metric_name, res) in enumerate(sig_results):
    ax_h = fig.add_subplot(gs_heat[idx_s])
    ax_h.set_facecolor("#1A1D2E")
    nemenyi_df = res["nemenyi"].copy()

    sns.heatmap(
        nemenyi_df, annot=True, fmt=".3f", ax=ax_h,
        cmap=cmap_nemenyi, vmin=0, vmax=1,
        linewidths=0.5, linecolor="#333",
        annot_kws={"size": 10, "color": "white"},
        cbar_kws={"shrink": 0.75, "pad": 0.02},
    )
    ax_h.set_title(
        f"Post-hoc Nemenyi — {metric_name}\np < 0.05 → diferencia significativa",
        color="white", fontsize=10,
    )
    ax_h.tick_params(colors="white")
    plt.setp(ax_h.get_xticklabels(), color="white", fontsize=9, rotation=0)
    plt.setp(ax_h.get_yticklabels(), color="white", fontsize=9, rotation=0)


# ── Fila 4: Evolución R²  +  Tabla resumen ───────────────────────────
gs_bot = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_main[4], wspace=0.35, width_ratios=[1.6, 1])

# Evolución R² por fold
ax_ev = fig.add_subplot(gs_bot[0])
ax_ev.set_facecolor("#1A1D2E")
folds = np.arange(1, 11)
for model, data in zip(MODELS, [r2_conv1d, r2_mlp, r2_lstm]):
    ax_ev.plot(folds, data, marker="o", label=model,
               color=COLORS[model], linewidth=2, markersize=6)
    ax_ev.fill_between(folds, data, alpha=0.08, color=COLORS[model])
ax_ev.axhline(y=0, color="#888", linestyle="--", alpha=0.6, linewidth=1)
ax_ev.set_xlabel("Fold", fontsize=10, color="#ccc")
ax_ev.set_ylabel("R² medio del fold", fontsize=10, color="#ccc")
ax_ev.set_title("Estabilidad del R² a través de los 10 folds", color="white", fontsize=11)
ax_ev.legend(fontsize=10, facecolor="#1A1D2E", labelcolor="white", edgecolor="#555")
ax_ev.set_xticks(folds)
ax_ev.grid(alpha=0.25, color="#555")

# Tabla resumen
ax_tbl = fig.add_subplot(gs_bot[1])
ax_tbl.set_facecolor("#1A1D2E")
ax_tbl.axis("off")

summary_rows = []
for metric_name, res in results.items():
    for m in MODELS:
        summary_rows.append({
            "Métrica": metric_name,
            "Modelo": m,
            "Media": f"{res['means'][m]:.3f}",
            "Rango": f"{res['mean_ranks'][m]:.2f}",
        })

col_labels  = ["Métrica", "Modelo", "Media", "Rango"]
table_data  = [[r["Métrica"], r["Modelo"], r["Media"], r["Rango"]] for r in summary_rows]

tbl = ax_tbl.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#444")
    if row == 0:
        cell.set_facecolor("#2E86AB")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        model_in_row = table_data[row - 1][1]
        cell.set_facecolor(COLORS.get(model_in_row, "#1A1D2E"))
        cell.set_text_props(color="white")
        cell.set_alpha(0.7)

ax_tbl.set_title("Resumen de métricas", color="white", fontsize=10, pad=10)

output_path = "friedman_ts_report.png"
plt.savefig(output_path, dpi=150, bbox_inches="tight",
            facecolor="#0F1117", edgecolor="none")
print(f"\n  Figura guardada en: {output_path}")
plt.close()

# ─────────────────────────────────────────────
# 6. RESUMEN FINAL EN TABLA PANDAS
# ─────────────────────────────────────────────
print("\n" + "=" * 72)
print("  TABLA RESUMEN ESTADÍSTICO COMPLETO")
print("=" * 72)

rows = []
for metric_name, res in results.items():
    for m in MODELS:
        rows.append({
            "Métrica": metric_name,
            "Modelo": m,
            "Media": round(res['means'][m], 4),
            "Std": round(res['stds'][m], 4),
            "Rango_Friedman": round(res['mean_ranks'][m], 3),
            "Friedman_chi2": round(res['friedman_stat'], 4),
            "p_valor": round(res['p_value'], 6),
            "Significativo": "Sí" if res['significant'] else "No",
        })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
print()

  PRUEBA DE FRIEDMAN — COMPARACIÓN DE MODELOS DE SERIES TEMPORALES
  Conv1D_Original | MLP_Light_Dropout | LSTM_Original
  10-fold Cross-Validation | 4 variables objetivo

────────────────────────────────────────────────────────────────────────
  MÉTRICA: R²
────────────────────────────────────────────────────────────────────────
  Estadístico de Friedman χ²  : 15.8000
  p-valor                     : 0.000371  ***
  Significativo (α=0.05)      : SÍ ✔

  Modelo             Media       Std   Rango medio
  ──────────────────────────────────────────────────
  Conv1D            0.9481    0.0178         3.000
  MLP               0.2164    0.4151         1.300
  LSTM              0.3079    0.3505         1.700

  Post-hoc Nemenyi (p-valores):
    Conv1D vs MLP: p=0.0004  ***
    Conv1D vs LSTM: p=0.0102  *
    MLP vs LSTM: p=0.6438  ns

  Wilcoxon signed-rank (pairwise):
    Conv1D vs MLP: W=0.0, p=0.0020  **
    Conv1D vs LSTM: W=0.0, p=0.0020  **
    MLP vs LSTM: W=9.0, p=0.0645  ns

──────

La prueba de Friedman, aplicada sobre los resultados de validación cruzada de 10 folds para las tres arquitecturas evaluadas Conv1D, MLP y LSTM, evidenció diferencias estadísticamente significativas en todas las métricas analizadas (R², MAE, MAPE y Gap de generalización), con un estadístico χ² = 15.80 y p = 0.000371 para las métricas de precisión, y χ² = 15.20, p = 0.0005 para el Gap, rechazando en todos los casos la hipótesis nula de equivalencia entre modelos con un nivel de confianza del 99.9%. El análisis post-hoc de Nemenyi confirmó que Conv1D obtuvo el rango de Friedman máximo posible (3.0/3.0) en las tres métricas de error R², MAE y MAPE, con diferencias significativas frente a MLP (p = 0.0004) y frente a LSTM (p = 0.0102), mientras que MLP y LSTM no se diferenciaron significativamente entre sí (p = 0.644), lo que ubica a Conv1D como una clase de rendimiento estadísticamente superior y aislada de las otras dos arquitecturas. En términos de magnitud, Conv1D alcanzó un R² promedio de 0.948 ± 0.018 a lo largo de los 10 folds, frente a 0.216 ± 0.415 del MLP y 0.308 ± 0.350 del LSTM; su MAE promedio fue de 0.800 unidades, correspondiente a una reducción del 77.2% respecto al MLP (3.507) y del 75.0% respecto al LSTM (3.204); y su MAPE promedio de 1.91% contrasta con el 7.69% del MLP y el 7.30% del LSTM. La evolución del R² a través de los folds reveló además que Conv1D mantiene una trayectoria estable y contenida en el rango [0.907, 0.972], sin caídas a valores negativos, a diferencia de MLP y LSTM que exhibieron folds con R² negativos (mínimos de −0.541 y −0.327 respectivamente), indicativo de colapso predictivo en segmentos temporales específicos del conjunto de datos. Finalmente, el análisis del Gap de generalización definido como (MAE_val − MAE_train) / MAE_train mostró que Conv1D presentó el menor sobreajuste sistemático (Gap promedio = 0.223), estadísticamente inferior al del LSTM (Gap = 2.199, p = 0.002 Wilcoxon), mientras que la comparación Conv1D vs MLP no alcanzó significancia en esta métrica (p = 0.131), sugiriendo que ambos generalizan de forma comparable aunque Conv1D lo hace con una precisión cuatro veces superior. En conjunto, la convergencia de evidencia estadística no paramétrica, magnitud de error, estabilidad inter-fold y capacidad de generalización posiciona a Conv1D como la arquitectura óptima para el modelado predictivo multivariante de temperatura y humedad en este contexto.